In [1]:
!pip install feast scikit-learn joblib --quiet

In [2]:
import sys
!{sys.executable} -m pip install feast scikit-learn joblib --quiet

In [1]:
import sys
!{sys.executable} -m pip install pyarrow --upgrade --quiet

In [2]:
import sys
print(sys.executable)
import feast
print(feast.__version__)

/opt/micromamba/envs/jupyterlab/bin/python3
0.64.0


/opt/micromamba/envs/jupyterlab/lib/python3.12/site-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [3]:
!feast version

/opt/micromamba/envs/jupyterlab/lib/python3.12/pty.py:95: DeprecationWarning: This process (pid=8329) is multi-threaded, use of forkpty() may lead to deadlocks in the child.
  pid, fd = os.forkpty()


Feast SDK Version: "0.64.0"


In [4]:
import os

REPO_ROOT  = os.getcwd()                          # wherever your notebook lives
FEAST_REPO = os.path.join(REPO_ROOT, 'feature_repo')
DATA_DIR   = os.path.join(FEAST_REPO, 'data')

In [5]:
os.makedirs(DATA_DIR, exist_ok=True)

print(f"Repo root  : {REPO_ROOT}")
print(f"Feast repo : {FEAST_REPO}")
print(f"Data dir   : {DATA_DIR}")

Repo root  : /home/jupyter
Feast repo : /home/jupyter/feature_repo
Data dir   : /home/jupyter/feature_repo/data


In [6]:
%%writefile feature_repo/feature_store.yaml
project: iris_feature_store
provider: local
registry: data/registry.db
online_store:
    type: sqlite
    path: data/online_store.db
offline_store:
    type: file
entity_key_serialization_version: 3

Overwriting feature_repo/feature_store.yaml


In [7]:
!find feature_repo -not -path '*/.*' | sort

feature_repo
feature_repo/__pycache__
feature_repo/__pycache__/iris_repo.cpython-312.pyc
feature_repo/data
feature_repo/data/iris_data_adapted_for_feast.csv
feature_repo/data/iris_data_adapted_for_feast.parquet
feature_repo/data/online_store.db
feature_repo/data/registry.db
feature_repo/feature_store.yaml
feature_repo/iris_repo.py


In [8]:
import shutil
import pandas as pd

In [9]:
csv_src = os.path.join(REPO_ROOT, 'iris_data_adapted_for_feast.csv')
csv_dst = os.path.join(DATA_DIR,  'iris_data_adapted_for_feast.csv')
parquet = os.path.join(DATA_DIR,  'iris_data_adapted_for_feast.parquet')

In [10]:
if not os.path.exists(csv_dst):
    shutil.copy(csv_src, csv_dst)
    print(f"Copied CSV → {csv_dst}")

In [11]:
df = pd.read_csv(csv_dst)

/opt/micromamba/envs/jupyterlab/lib/python3.12/site-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [12]:
df['event_timestamp']   = pd.to_datetime(df['event_timestamp'],   utc=True)
df['created_timestamp'] = pd.to_datetime(df['created_timestamp'], utc=True)
df.to_parquet(parquet, index=False)

/opt/micromamba/envs/jupyterlab/lib/python3.12/site-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [13]:
print(f"Shape   : {df.shape}")
print(f"Columns : {list(df.columns)}")
print()
print(df.head(6).to_string())

Shape   : (45, 8)
Columns : ['event_timestamp', 'iris_id', 'sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'species', 'created_timestamp']

                   event_timestamp  iris_id  sepal_length  sepal_width  petal_length  petal_width     species                created_timestamp
0 2025-09-17 10:40:17.102131+00:00     1001          5.52         2.53          3.86         1.13  versicolor 2025-10-02 10:40:17.172178+00:00
1 2025-09-18 10:40:17.102131+00:00     1001          5.50         2.24          3.60         1.08  versicolor 2025-10-02 10:40:17.172178+00:00
2 2025-09-19 10:40:17.102131+00:00     1001          5.55         2.47          3.75         1.08  versicolor 2025-10-02 10:40:17.172178+00:00
3 2025-09-20 10:40:17.102131+00:00     1001          5.45         2.37          3.92         1.20  versicolor 2025-10-02 10:40:17.172178+00:00
4 2025-09-21 10:40:17.102131+00:00     1001          5.65         2.52          3.95         1.17  versicolor 2025-10-02 10:40:17.1

/opt/micromamba/envs/jupyterlab/lib/python3.12/site-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [14]:
%%writefile feature_repo/iris_repo.py
"""
Feast Feature Repository Definitions — IRIS Pipeline
Single source of truth for all feature definitions.
Used by BOTH training (offline) and inference (online) paths.
"""
from datetime import timedelta
from feast import Entity, FeatureView, Field, FileSource, ValueType
from feast.types import Float32, String

# 1. DATA SOURCE
# Points Feast at the Parquet file holding raw IRIS features.
# timestamp_field is required by Feast for point-in-time joins.
iris_source = FileSource(
    name="iris_source",
    path="data/iris_data_adapted_for_feast.parquet",
    timestamp_field="event_timestamp",
    created_timestamp_column="created_timestamp",
)

# 2. ENTITY
# The domain object the features describe.
# iris_id is the join key Feast uses to find the right feature row
# for a given iris plant in both offline and online retrieval.
iris = Entity(
    name="iris_id",
    value_type=ValueType.INT64,
    description="Unique identifier for each iris plant / measurement sample",
)

# 3. FEATURE VIEW
# Groups the 4 IRIS measurement columns + species label under one name,
# tied to the iris_id entity and the FileSource above.
# TTL = 365 days means a value stays valid for 365 days after its event_timestamp.
iris_feature_view = FeatureView(
    name="iris_features",
    entities=[iris],
    ttl=timedelta(days=365),
    schema=[
        Field(name="sepal_length", dtype=Float32),
        Field(name="sepal_width",  dtype=Float32),
        Field(name="petal_length", dtype=Float32),
        Field(name="petal_width",  dtype=Float32),
        Field(name="species",      dtype=String),
    ],
    online=True,
    source=iris_source,
    tags={"team": "mlops_week3", "dataset": "iris"},
)

Overwriting feature_repo/iris_repo.py


In [15]:
import sys
sys.path.insert(0, FEAST_REPO)
from iris_repo import iris_source, iris, iris_feature_view

print("=== DATA SOURCE ===")
print(f"  Name            : {iris_source.name}")
print(f"  Path            : {iris_source.path}")
print(f"  Timestamp field : {iris_source.timestamp_field}")
print(f"  Created ts col  : {iris_source.created_timestamp_column}")

print("\n=== ENTITY ===")
print(f"  Name        : {iris.name}")
print(f"  Join key    : {iris.join_key}")
print(f"  Value type  : {iris.value_type}")
print(f"  Description : {iris.description}")

print("\n=== FEATURE VIEW ===")
print(f"  Name     : {iris_feature_view.name}")
print(f"  Entities : {iris_feature_view.entities}")
print(f"  TTL      : {iris_feature_view.ttl}")
print(f"  Online   : {iris_feature_view.online}")
print("  Schema:")
for f in iris_feature_view.schema:
    print(f"    - {f.name:<14} {f.dtype}")

=== DATA SOURCE ===
  Name            : iris_source
  Path            : data/iris_data_adapted_for_feast.parquet
  Timestamp field : event_timestamp
  Created ts col  : created_timestamp

=== ENTITY ===
  Name        : iris_id
  Join key    : iris_id
  Value type  : ValueType.INT64
  Description : Unique identifier for each iris plant / measurement sample

=== FEATURE VIEW ===
  Name     : iris_features
  Entities : ['iris_id']
  TTL      : 365 days, 0:00:00
  Online   : True
  Schema:
    - sepal_width    Float32
    - species        String
    - sepal_length   Float32
    - petal_length   Float32
    - petal_width    Float32


# Task 3

In [16]:
!cd feature_repo && feast apply

/opt/micromamba/envs/jupyterlab/lib/python3.12/pty.py:95: DeprecationWarning: This process (pid=8329) is multi-threaded, use of forkpty() may lead to deadlocks in the child.
  pid, fd = os.forkpty()


No project found in the repository. Using project name iris_feature_store defined in feature_store.yaml
Applying changes for project iris_feature_store
Updated feature view iris_features
	batch_source: type: BATCH_FILE
timestamp_field: "event_timestamp"
created_timestamp_column: "created_timestamp"
file_options {
  uri: "data/iris_data_adapted_for_feast.parquet"
}
data_source_class_type: "feast.infra.offline_stores.file_source.FileSource"
name: "iris_source"
meta {
  created_timestamp {
    seconds: 1783173655
    nanos: 354506000
  }
  last_updated_timestamp {
    seconds: 1783173656
    nanos: 112689000
  }
}
 -> type: BATCH_FILE
timestamp_field: "event_timestamp"
created_timestamp_column: "created_timestamp"
file_options {
  uri: "data/iris_data_adapted_for_feast.parquet"
}
data_source_class_type: "feast.infra.offline_stores.file_source.FileSource"
name: "iris_source"
meta {
  created_timestamp {
    seconds: 1783174248
    nanos: 909884000
  }
  last_updated_timestamp {
    seconds

In [17]:
from datetime import datetime, timezone

end_ts = datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%S')
print(f"Materializing features up to: {end_ts}")

!cd feature_repo && feast materialize-incremental {end_ts}

Materializing features up to: 2026-07-04T14:10:50
Materializing 1 feature views to 2026-07-04 14:10:50+00:00 into the sqlite online store.

iris_features from 2026-07-04 14:01:18+00:00 to 2026-07-04 14:10:50+00:00:


In [18]:
import sqlite3

db_path = os.path.join(DATA_DIR, 'online_store.db')
con     = sqlite3.connect(db_path)

tables = con.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()
print("Online store tables:", [t[0] for t in tables])

rows = con.execute(
    "SELECT * FROM 'iris_feature_store_iris_features'"
).fetchall()
print(f"\nRows in online store: {len(rows)}")
for r in rows:
    print(r)

con.close()

Online store tables: ['iris_feature_store_iris_features']

Rows in online store: 15
(b'\x01\x00\x00\x00\x02\x00\x00\x00\x07\x00\x00\x00iris_id\x04\x00\x00\x00\x08\x00\x00\x00\xe9\x03\x00\x00\x00\x00\x00\x00', 'sepal_length', b'5ff\xae@', None, 1759315217, 1759401617)
(b'\x01\x00\x00\x00\x02\x00\x00\x00\x07\x00\x00\x00iris_id\x04\x00\x00\x00\x08\x00\x00\x00\xe9\x03\x00\x00\x00\x00\x00\x00', 'sepal_width', b'5=\n\x17@', None, 1759315217, 1759401617)
(b'\x01\x00\x00\x00\x02\x00\x00\x00\x07\x00\x00\x00iris_id\x04\x00\x00\x00\x08\x00\x00\x00\xe9\x03\x00\x00\x00\x00\x00\x00', 'petal_length', b'5\x8f\xc2u@', None, 1759315217, 1759401617)
(b'\x01\x00\x00\x00\x02\x00\x00\x00\x07\x00\x00\x00iris_id\x04\x00\x00\x00\x08\x00\x00\x00\xe9\x03\x00\x00\x00\x00\x00\x00', 'petal_width', b'5\x1f\x85\x8b?', None, 1759315217, 1759401617)
(b'\x01\x00\x00\x00\x02\x00\x00\x00\x07\x00\x00\x00iris_id\x04\x00\x00\x00\x08\x00\x00\x00\xe9\x03\x00\x00\x00\x00\x00\x00', 'species', b'\x12\nversicolor', None, 175931521

In [19]:
!cd feature_repo && feast feature-views list

NAME           ENTITIES     TYPE         ENABLED    STATE
iris_features  {'iris_id'}  FeatureView  Yes        AVAILABLE_ONLINE


# Task 4

In [20]:
import pandas as pd
import feast
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder
from joblib import dump

fs = feast.FeatureStore(repo_path=FEAST_REPO)
print(f"Connected to Feast project: {fs.project}")

Connected to Feast project: iris_feature_store


/opt/micromamba/envs/jupyterlab/lib/python3.12/site-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [21]:
# Read from CSV directly instead of parquet to avoid pyarrow version conflict
raw_df = pd.read_csv(csv_dst)
raw_df['event_timestamp'] = pd.to_datetime(raw_df['event_timestamp'], utc=True)
raw_df['created_timestamp'] = pd.to_datetime(raw_df['created_timestamp'], utc=True)

entity_df = raw_df[['iris_id', 'event_timestamp']].copy()

print(f"Entity dataframe shape: {entity_df.shape}")
print(entity_df.head(6).to_string())

Entity dataframe shape: (45, 2)
   iris_id                  event_timestamp
0     1001 2025-09-17 10:40:17.102131+00:00
1     1001 2025-09-18 10:40:17.102131+00:00
2     1001 2025-09-19 10:40:17.102131+00:00
3     1001 2025-09-20 10:40:17.102131+00:00
4     1001 2025-09-21 10:40:17.102131+00:00
5     1001 2025-09-22 10:40:17.102131+00:00


/opt/micromamba/envs/jupyterlab/lib/python3.12/site-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [22]:
parquet = csv_dst
print(f"Using data file: {parquet}")

Using data file: /home/jupyter/feature_repo/data/iris_data_adapted_for_feast.csv


In [23]:
FEATURES = [
    'iris_features:sepal_length',
    'iris_features:sepal_width',
    'iris_features:petal_length',
    'iris_features:petal_width',
    'iris_features:species',
]

print("Fetching historical features from Feast OFFLINE store...")
training_df = fs.get_historical_features(
    entity_df=entity_df,
    features=FEATURES,
).to_df()

print(f"Training dataframe shape: {training_df.shape}")
print(f"Columns: {list(training_df.columns)}")
print()
print(training_df.head(8).to_string())

Fetching historical features from Feast OFFLINE store...


/opt/micromamba/envs/jupyterlab/lib/python3.12/site-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Training dataframe shape: (45, 7)
Columns: ['iris_id', 'event_timestamp', 'sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'species']

   iris_id                  event_timestamp  sepal_length  sepal_width  petal_length  petal_width     species
0     1001 2025-09-17 10:40:17.102131+00:00          5.52         2.53          3.86         1.13  versicolor
1     1003 2025-09-17 10:40:17.102131+00:00          5.09         3.42          1.34         0.22      setosa
2     1002 2025-09-17 10:40:17.102131+00:00          4.92         3.05          1.43         0.29      setosa
3     1001 2025-09-18 10:40:17.102131+00:00          5.50         2.24          3.60         1.08  versicolor
4     1002 2025-09-18 10:40:17.102131+00:00          5.05         2.94          1.53         0.31      setosa
5     1003 2025-09-18 10:40:17.102131+00:00          5.24         3.44          1.56         0.05      setosa
6     1001 2025-09-19 10:40:17.102131+00:00          5.55         2.47          3.

/opt/micromamba/envs/jupyterlab/lib/python3.12/site-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [24]:
FEATURE_COLS = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']

X  = training_df[FEATURE_COLS]
le = LabelEncoder()
y  = le.fit_transform(training_df['species'])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
acc    = accuracy_score(y_test, y_pred)

print(f"Training samples : {len(X_train)}")
print(f"Test samples     : {len(X_test)}")
print(f"Test accuracy    : {acc:.4f}")
print()
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=le.classes_))

Training samples : 36
Test samples     : 9
Test accuracy    : 1.0000

Classification Report:
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00         6
  versicolor       1.00      1.00      1.00         3

    accuracy                           1.00         9
   macro avg       1.00      1.00      1.00         9
weighted avg       1.00      1.00      1.00         9



/opt/micromamba/envs/jupyterlab/lib/python3.12/site-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [25]:
model_path   = os.path.join(REPO_ROOT, 'iris_model.joblib')
encoder_path = os.path.join(REPO_ROOT, 'iris_label_encoder.joblib')

dump(model, model_path)
dump(le,    encoder_path)

print(f"Model saved   : {model_path}")
print(f"Encoder saved : {encoder_path}")

Model saved   : /home/jupyter/iris_model.joblib
Encoder saved : /home/jupyter/iris_label_encoder.joblib


# Task 5

In [26]:
from joblib import load

model = load(os.path.join(REPO_ROOT, 'iris_model.joblib'))
le    = load(os.path.join(REPO_ROOT, 'iris_label_encoder.joblib'))
fs    = feast.FeatureStore(repo_path=FEAST_REPO)

FEATURES = [
    'iris_features:sepal_length',
    'iris_features:sepal_width',
    'iris_features:petal_length',
    'iris_features:petal_width',
    'iris_features:species',
]
FEATURE_COLS = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']

print("Model loaded successfully")
print(f"Label classes: {le.classes_}")

Model loaded successfully
Label classes: ['setosa' 'versicolor']


In [27]:
def predict_iris(iris_ids):
    entity_rows = [{'iris_id': i} for i in iris_ids]

    online_resp = fs.get_online_features(
        features=FEATURES,
        entity_rows=entity_rows,
    ).to_df()

    preds = model.predict(online_resp[FEATURE_COLS])
    online_resp['predicted_species'] = le.inverse_transform(preds)
    return online_resp

result = predict_iris([1001, 1002, 1003])

print("=== Online (Real-Time) Inference Results ===")
print(result[['iris_id', *FEATURE_COLS, 'species', 'predicted_species']].to_string(index=False))

=== Online (Real-Time) Inference Results ===
 iris_id  sepal_length  sepal_width  petal_length  petal_width    species predicted_species
    1001          5.45         2.36          3.84         1.09 versicolor        versicolor
    1002          4.84         2.90          1.29         0.20     setosa            setosa
    1003          4.85         3.40          1.19         0.29     setosa            setosa


/opt/micromamba/envs/jupyterlab/lib/python3.12/site-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [28]:
raw    = pd.read_csv(csv_dst)
raw['event_timestamp'] = pd.to_datetime(raw['event_timestamp'])
latest = raw.sort_values('event_timestamp').groupby('iris_id').tail(1)

print("=== Ground Truth (latest raw CSV row per iris_id) ===")
print(latest[['iris_id', *FEATURE_COLS, 'species']].to_string(index=False))

print()

merged = result[['iris_id', 'species', 'predicted_species']].merge(
    latest[['iris_id', 'species']].rename(columns={'species': 'raw_species'}),
    on='iris_id'
)
merged['features_match'] = merged['species'] == merged['raw_species']
merged['pred_correct']   = merged['predicted_species'] == merged['raw_species']

print("=== Consistency Check ===")
print(merged.to_string(index=False))
print()
print(f"Feast online features match raw CSV : {merged['features_match'].all()}")
print(f"All predictions correct             : {merged['pred_correct'].all()}")

=== Ground Truth (latest raw CSV row per iris_id) ===
 iris_id  sepal_length  sepal_width  petal_length  petal_width    species
    1001          5.45         2.36          3.84         1.09 versicolor
    1002          4.84         2.90          1.29         0.20     setosa
    1003          4.85         3.40          1.19         0.29     setosa

=== Consistency Check ===
 iris_id    species predicted_species raw_species  features_match  pred_correct
    1001 versicolor        versicolor  versicolor            True          True
    1002     setosa            setosa      setosa            True          True
    1003     setosa            setosa      setosa            True          True

Feast online features match raw CSV : True
All predictions correct             : True


/opt/micromamba/envs/jupyterlab/lib/python3.12/site-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [29]:
!cd /home/jupyter && git status

On branch main
Changes not staged for commit:
  (use "git add/rm <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	deleted:    22DP1000105_Assignment_1_MAY_2026_MLOps.ipynb
	deleted:    README.md
	deleted:    data.csv
	deleted:    notebook_template.ipynb
	deleted:    week-1.py

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	.gitconfig
	22DP1000105.ipynb
	feature_repo/
	iris_data_adapted_for_feast.csv
	iris_label_encoder.joblib
	iris_model.joblib

no changes added to commit (use "git add" and/or "git commit -a")


/opt/micromamba/envs/jupyterlab/lib/python3.12/pty.py:95: DeprecationWarning: This process (pid=8329) is multi-threaded, use of forkpty() may lead to deadlocks in the child.
  pid, fd = os.forkpty()


In [30]:
!cd /home/jupyter && git checkout -b week_3

Switched to a new branch 'week_3'


In [38]:
!cd /home/jupyter && git add 22DP1000105_Assignment_3_MAY_2026_MLOps.ipynb

In [39]:
!cd /home/jupyter && git add iris_data_adapted_for_feast.csv

In [40]:
!cd /home/jupyter && git add feature_repo/feature_store.yaml
!cd /home/jupyter && git add feature_repo/iris_repo.py

In [41]:
!cd /home/jupyter && git add feature_repo/data/iris_data_adapted_for_feast.csv
!cd /home/jupyter && git add feature_repo/data/iris_data_adapted_for_feast.parquet

In [42]:
!cd /home/jupyter && git status

On branch week_3
Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	new file:   22DP1000105.ipynb
	new file:   22DP1000105_Assignment_3_MAY_2026_MLOps.ipynb
	new file:   feature_repo/data/iris_data_adapted_for_feast.csv
	new file:   feature_repo/data/iris_data_adapted_for_feast.parquet
	new file:   feature_repo/feature_store.yaml
	new file:   feature_repo/iris_repo.py
	new file:   iris_data_adapted_for_feast.csv

Changes not staged for commit:
  (use "git add/rm <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	deleted:    22DP1000105.ipynb
	deleted:    22DP1000105_Assignment_1_MAY_2026_MLOps.ipynb
	deleted:    README.md
	deleted:    data.csv
	deleted:    notebook_template.ipynb
	deleted:    week-1.py

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	.gitconfig
	feature_repo/__pycache__/
	feature_repo/data/online_store.db
	feature_repo/data/registry.db
	iris_

In [43]:
!cd /home/jupyter && git commit -m "Week 3: Integrate Feast feature store into IRIS pipeline (Tasks 1-5)"

[week_3 20fb7cd] Week 3: Integrate Feast feature store into IRIS pipeline (Tasks 1-5)
 7 files changed, 2885 insertions(+)
 create mode 100644 22DP1000105.ipynb
 create mode 100644 22DP1000105_Assignment_3_MAY_2026_MLOps.ipynb
 create mode 100644 feature_repo/data/iris_data_adapted_for_feast.csv
 create mode 100644 feature_repo/data/iris_data_adapted_for_feast.parquet
 create mode 100644 feature_repo/feature_store.yaml
 create mode 100644 feature_repo/iris_repo.py
 create mode 100644 iris_data_adapted_for_feast.csv


In [ ]:
!cd /home/jupyter && git push origin week_3

Username for 'https://github.com': 

In [ ]:
Anurag0325